# `audit.ipynb` — exclusions investigation

This notebook runs on `data/reference/modeling_df.parquet` (or
`data/processed/modeling_df.parquet` when the fresh pipeline has been
built). It no longer requires the raw Kuopio archives or
`info_participants.xlsx`.

The §A detector-calibration overlay, §B `n_strikes_plate` scout, and
the per-subject-alpha + cohort-demographics appendices that previously
lived here were moved to `src/build_scout_assets.py`. Running that
script writes all of those outputs into `data/scout/`.


In [1]:
import os
import pathlib

import pandas as pd

os.makedirs("model_assets", exist_ok=True)
OUTPUT_PATH = "model_assets/audit/"
os.makedirs(OUTPUT_PATH, exist_ok=True)

PROCESSED = pathlib.Path("../data/processed/modeling_df.parquet")
REFERENCE = pathlib.Path("../data/reference/modeling_df.parquet")

if PROCESSED.exists():
    modeling_parquet = PROCESSED
else:
    print(
        "[warning] data/processed/modeling_df.parquet not found — "
        "falling back to committed frozen reference at data/reference/. "
        "Run bin/build.sh for the fresh pipeline output."
    )
    modeling_parquet = REFERENCE


[warning] data/processed/modeling_df.parquet not found — falling back to committed frozen reference at data/reference/. Run bin/build.sh for the fresh pipeline output.


## C. Exclusions investigation

Counts of `modeling_include = False` rows in `modeling_df.parquet` broken down by reason, sex, speed, and subject. Trials can carry multiple flags; each is counted independently. `pipeline_error:*` subtypes are collapsed to one family for top-level reporting.

Counts are recomputed directly from the parquet (cheap) and each breakdown is written as a CSV under `model_assets/audit/exclusions_*.csv`.


In [2]:
def _flag_set(raw: object) -> list[str]:
    if not isinstance(raw, str) or not raw.strip():
        return []
    return [f.strip() for f in raw.split(",") if f.strip()]


def _flag_family(flag: str) -> str:
    if flag.startswith("pipeline_error:"):
        return "pipeline_error"
    return flag


df = pd.read_parquet(modeling_parquet)

df["flags"] = df["qc_flags"].apply(_flag_set)
df["flag_families"] = df["flags"].apply(lambda flags: sorted({_flag_family(f) for f in flags}))
df["excluded"] = ~df["modeling_include"]

n_total = len(df)
n_excluded = int(df["excluded"].sum())
overall_rate = n_excluded / n_total

reason_counts = (
    df[df["excluded"]].explode("flag_families")["flag_families"]
    .value_counts().rename_axis("reason")
)
pipeline_error_detail = (
    df[df["excluded"]].explode("flags")
    .query("flags.str.startswith('pipeline_error:')", engine="python")["flags"]
    .value_counts()
)

sex_total = df.groupby("sex").size().rename("n_total")
sex_excluded = df[df["excluded"]].groupby("sex").size().rename("n_excluded")
sex_tbl = pd.concat([sex_total, sex_excluded], axis=1).fillna(0)
sex_tbl["n_excluded"] = sex_tbl["n_excluded"].astype(int)
sex_tbl["rate"] = sex_tbl["n_excluded"] / sex_tbl["n_total"]
f_rate = sex_tbl.loc["F", "rate"] if "F" in sex_tbl.index else 0.0
m_rate = sex_tbl.loc["M", "rate"] if "M" in sex_tbl.index else 0.0
fm_ratio = (f_rate / m_rate) if m_rate > 0 else float("inf")

speed_total = df.groupby("speed").size().rename("n_total")
speed_excluded = df[df["excluded"]].groupby("speed").size().rename("n_excluded")
speed_tbl = pd.concat([speed_total, speed_excluded], axis=1).fillna(0)
speed_tbl["n_excluded"] = speed_tbl["n_excluded"].astype(int)
speed_tbl["rate"] = speed_tbl["n_excluded"] / speed_tbl["n_total"]

sx_total = df.groupby(["sex", "speed"]).size().rename("n_total")
sx_excluded = df[df["excluded"]].groupby(["sex", "speed"]).size().rename("n_excluded")
sx_tbl = pd.concat([sx_total, sx_excluded], axis=1).fillna(0)
sx_tbl["n_excluded"] = sx_tbl["n_excluded"].astype(int)
sx_tbl["rate"] = sx_tbl["n_excluded"] / sx_tbl["n_total"]
sx_tbl["hot"] = sx_tbl["rate"] > 2 * overall_rate

subj_total = df.groupby("subject_id").size().rename("n_total")
subj_excluded = df[df["excluded"]].groupby("subject_id").size().rename("n_excluded")
subj_sex = df.groupby("subject_id")["sex"].first().rename("sex")
subj_tbl = pd.concat([subj_sex, subj_total, subj_excluded], axis=1)
subj_tbl = subj_tbl[subj_tbl["n_excluded"].notna()].copy()
subj_tbl["n_excluded"] = subj_tbl["n_excluded"].astype(int)
subj_tbl["rate"] = subj_tbl["n_excluded"] / subj_tbl["n_total"]
subj_tbl = subj_tbl.sort_values("n_excluded", ascending=False)
high_rate_subjects = subj_tbl[subj_tbl["rate"] > 0.10]

reason_sex = (
    df[df["excluded"]].explode("flag_families")
    .groupby(["flag_families", "sex"]).size().unstack(fill_value=0)
)


In [3]:
print(f"Total: {n_total} | Excluded: {n_excluded} ({overall_rate * 100:.2f}%)")
print("\nReasons:")
print(reason_counts.to_string())
if not pipeline_error_detail.empty:
    print("\npipeline_error subtypes:")
    print(pipeline_error_detail.to_string())
print("\nBy sex:")
print(sex_tbl.to_string())
print(f"F/M rate ratio: {fm_ratio:.2f}")
print("\nBy speed:")
print(speed_tbl.to_string())
print("\nSex \u00d7 speed:")
print(sx_tbl.to_string())
print("\nPer-subject (\u22651 exclusion):")
print(subj_tbl.to_string())
if not high_rate_subjects.empty:
    print(f"\n>10% rate subjects: {high_rate_subjects.index.tolist()}")
print("\nReason \u00d7 sex:")
print(reason_sex.to_string())

Total: 2818 | Excluded: 37 (1.31%)

Reasons:
reason
pipeline_error           19
n_strikes_below_3        15
imu_pelvis_no_cadence     3

pipeline_error subtypes:
flags
pipeline_error:OSError    19

By sex:
     n_total  n_excluded      rate
sex                               
F       1021           5  0.004897
M       1797          32  0.017807
F/M rate ratio: 0.28

By speed:
       n_total  n_excluded      rate
speed                               
comf       943          10  0.010604
fast       938          18  0.019190
slow       937           9  0.009605

Sex × speed:
           n_total  n_excluded      rate    hot
sex speed                                      
F   comf       342           1  0.002924  False
    fast       338           1  0.002959  False
    slow       341           3  0.008798  False
M   comf       601           9  0.014975  False
    fast       600          17  0.028333   True
    slow       596           6  0.010067  False

Per-subject (≥1 exclusion):
          

In [4]:
# Dump each audit table to model_assets/audit/ as a CSV. Rates stay as
# proportions (floats) — formatting is the writeup's job.
headline = pd.Series(
    {
        "n_total": n_total,
        "n_excluded": n_excluded,
        "overall_rate": overall_rate,
        "f_rate": f_rate,
        "m_rate": m_rate,
        "fm_ratio": fm_ratio,
        "hot_cells": str(sx_tbl[sx_tbl["hot"]].index.tolist()),
        "high_rate_subjects": str(high_rate_subjects.index.tolist()),
    },
    name="value",
)
headline.rename_axis("metric").to_csv(f"{OUTPUT_PATH}exclusions_headline.csv")

reason_counts.to_frame("n").to_csv(f"{OUTPUT_PATH}exclusions_by_reason.csv")
if not pipeline_error_detail.empty:
    pipeline_error_detail.to_frame("n").to_csv(
        f"{OUTPUT_PATH}exclusions_pipeline_error_subtypes.csv"
    )
sex_tbl.to_csv(f"{OUTPUT_PATH}exclusions_by_sex.csv")
speed_tbl.to_csv(f"{OUTPUT_PATH}exclusions_by_speed.csv")
sx_tbl.to_csv(f"{OUTPUT_PATH}exclusions_by_sex_speed.csv")
subj_tbl.to_csv(f"{OUTPUT_PATH}exclusions_by_subject.csv")
reason_sex.to_csv(f"{OUTPUT_PATH}exclusions_reason_by_sex.csv")
high_rate_subjects.to_csv(f"{OUTPUT_PATH}exclusions_high_rate_subjects.csv")
print(f"Wrote exclusions CSVs to {OUTPUT_PATH}")


Wrote exclusions CSVs to model_assets/audit/
